# 🎬 YouTube Automation Tool
**Faceless video generator — 100% free**

### How to use:
1. Run **Cell 1** — installs everything (takes ~1 min)
2. Fill in **Cell 2** — your topic, script, API key
3. Run **Cell 3** — generates the video
4. Run **Cell 4** — downloads video + thumbnail to your phone

> 💡 Get a **free** Pexels API key at [pexels.com/api](https://www.pexels.com/api/) — takes 30 seconds

In [ ]:
# ── CELL 1: Install dependencies ──────────────────────────────────────────────
print('Installing packages... (takes about 1 minute)')
!pip install edge-tts moviepy Pillow requests imageio imageio-ffmpeg -q
!apt-get install -y ffmpeg > /dev/null 2>&1
print('✅ Done! Run the next cell.')

In [ ]:
# ── CELL 2: Your settings ─────────────────────────────────────────────────────
# Fill in these fields, then run Cell 3

TOPIC = "The Fall of the Roman Empire"   # ← change this

SCRIPT = """At its peak, the Roman Empire controlled over five million square kilometers.
It was the most powerful civilization the world had ever seen.
But in 476 AD, it all came crashing down.
So what went wrong? How did the greatest empire in history simply... collapse?
The answer involves a mix of military overreach, economic crisis, political corruption,
and a series of devastating invasions that Rome could no longer hold back.
This is the story of how an empire dies."""
# ↑ Replace with your own script. Keep ### at the end to mark the boundary.

VOICE = "narrator_male"   # options: narrator_male, narrator_female, male_us, female_us, male_uk, female_uk

PEXELS_API_KEY = ""      # ← paste your free Pexels key here (or leave empty to skip B-roll)

print(f'Topic  : {TOPIC}')
print(f'Voice  : {VOICE}')
print(f'Script : {len(SCRIPT.split())} words')
print(f'Pexels : {"✅ key set" if PEXELS_API_KEY else "⚠️  no key — video will use black background"}')

In [ ]:
# ── CELL 3: Generate video ────────────────────────────────────────────────────
import asyncio, json, os, shutil, requests
from pathlib import Path
from io import BytesIO

# ── TTS ───────────────────────────────────────────────────────────────────────
import edge_tts

VOICES = {
    "narrator_male":   "en-US-ChristopherNeural",
    "narrator_female": "en-US-JennyNeural",
    "male_us":         "en-US-GuyNeural",
    "female_us":       "en-US-AriaNeural",
    "male_uk":         "en-GB-RyanNeural",
    "female_uk":       "en-GB-SoniaNeural",
}

async def _tts(text, voice, path):
    words = []
    comm = edge_tts.Communicate(text, voice)
    with open(path, 'wb') as f:
        async for chunk in comm.stream():
            if chunk['type'] == 'audio':
                f.write(chunk['data'])
            elif chunk['type'] == 'WordBoundary':
                words.append({
                    'word':     chunk['text'],
                    'start':    chunk['offset']   / 10_000_000,
                    'duration': chunk['duration'] / 10_000_000,
                })
    return words

# ── B-roll ────────────────────────────────────────────────────────────────────
def _pexels_clips(query, key, n=2):
    r = requests.get('https://api.pexels.com/videos/search',
                     headers={'Authorization': key},
                     params={'query': query, 'per_page': n, 'orientation': 'landscape'},
                     timeout=15)
    if r.status_code != 200: return []
    urls = []
    for v in r.json().get('videos', []):
        files = v.get('video_files', [])
        hd = [f for f in files if f.get('quality') == 'hd' and f.get('width', 0) >= 1280]
        pick = (hd or files)
        if pick: urls.append(pick[0]['link'])
    return urls

def _download(url, path):
    try:
        r = requests.get(url, stream=True, timeout=60)
        with open(path, 'wb') as f:
            for chunk in r.iter_content(8192): f.write(chunk)
        return True
    except: return False

def get_broll(topic, script, key, tmp, count=6):
    kw = [w.lower() for w in topic.split() if len(w) > 3][:3]
    clips = []
    for i, k in enumerate(kw):
        for j, url in enumerate(_pexels_clips(k, key)):
            p = tmp / f'b_{i}_{j}.mp4'
            if _download(url, p): clips.append(p)
            if len(clips) >= count: return clips
    return clips

# ── Video assembly ────────────────────────────────────────────────────────────
def make_video(broll, audio_path, words, out):
    from moviepy.editor import (
        VideoFileClip, AudioFileClip, ColorClip, TextClip,
        concatenate_videoclips, CompositeVideoClip
    )
    SIZE = (1920, 1080)
    audio = AudioFileClip(str(audio_path))
    total = audio.duration

    bg_clips, cur, idx = [], 0.0, 0
    if broll:
        while cur < total:
            p = broll[idx % len(broll)]
            try:
                c = VideoFileClip(str(p)).without_audio().resize(SIZE)
                rem = total - cur
                if c.duration > rem: c = c.subclip(0, rem)
                bg_clips.append(c.set_start(cur))
                cur += c.duration
            except: pass
            idx += 1
            if idx > len(broll) * 3: break

    if not bg_clips:
        bg_clips = [ColorClip(SIZE, color=(10, 10, 25), duration=total)]

    bg = concatenate_videoclips(bg_clips, method='compose').set_audio(audio)

    # subtitles
    subs = []
    chunk = []
    for w in words:
        chunk.append(w)
        if len(chunk) == 7:
            subs.append(chunk); chunk = []
    if chunk: subs.append(chunk)

    sub_clips = []
    for g in subs:
        text  = ' '.join(w['word'] for w in g)
        start = g[0]['start']
        end   = min(g[-1]['start'] + g[-1]['duration'], total)
        dur   = end - start
        if dur <= 0: continue
        try:
            tc = (TextClip(text, fontsize=56, color='white', font='DejaVu-Sans-Bold',
                           stroke_color='black', stroke_width=2,
                           method='caption', size=(1600, None))
                  .set_start(start).set_duration(dur)
                  .set_position(('center', 880)))
            sub_clips.append(tc)
        except: pass

    final = CompositeVideoClip([bg] + sub_clips, size=SIZE)
    final.write_videofile(str(out), fps=30, codec='libx264',
                          audio_codec='aac', preset='medium',
                          threads=2, logger=None)
    audio.close()

# ── Thumbnail ─────────────────────────────────────────────────────────────────
def make_thumbnail(title, out):
    from PIL import Image, ImageDraw, ImageFont, ImageEnhance
    SIZE = (1280, 720)
    img  = Image.new('RGB', SIZE, (15, 15, 35))
    draw = ImageDraw.Draw(img)
    for y in range(SIZE[1] // 2, SIZE[1]):
        draw.line([(0, y), (SIZE[0], y)], fill=(0, 0, 0))
    try:
        font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 72)
    except:
        font = ImageFont.load_default()
    words = title.upper().split()
    lines, line = [], []
    for w in words:
        line.append(w)
        if draw.textbbox((0,0),' '.join(line),font=font)[2] > SIZE[0]-100 and len(line)>1:
            line.pop(); lines.append(' '.join(line)); line=[w]
    if line: lines.append(' '.join(line))
    y = (SIZE[1] - len(lines)*82)//2 + 80
    for text in lines:
        bb = draw.textbbox((0,0), text, font=font)
        x  = (SIZE[0] - (bb[2]-bb[0])) // 2
        draw.text((x+3, y+3), text, fill=(0,0,0), font=font)
        draw.text((x, y),     text, fill=(255,220,50), font=font)
        y += 82
    img.save(str(out), 'JPEG', quality=95)

# ── RUN PIPELINE ─────────────────────────────────────────────────────────────
safe   = ''.join(c if c.isalnum() else '_' for c in TOPIC)[:35].lower()
tmp    = Path('/content/tmp'); tmp.mkdir(exist_ok=True)
out_mp4   = Path(f'/content/{safe}.mp4')
out_thumb = Path(f'/content/{safe}_thumbnail.jpg')

print('🎙  Generating voiceover...')
voice = VOICES.get(VOICE, VOICES['narrator_male'])
words = asyncio.run(_tts(SCRIPT, voice, tmp / 'audio.mp3'))
print(f'   ✅ {len(words)} word timestamps')

broll = []
if PEXELS_API_KEY:
    print('🎬  Fetching B-roll...')
    broll = get_broll(TOPIC, SCRIPT, PEXELS_API_KEY, tmp)
    print(f'   ✅ {len(broll)} clips')
else:
    print('⚠️   No Pexels key — using dark background')

print('🎞   Assembling video (this takes a few minutes)...')
make_video(broll, tmp / 'audio.mp3', words, out_mp4)
print(f'   ✅ {out_mp4}')

print('🖼   Creating thumbnail...')
make_thumbnail(TOPIC, out_thumb)
print(f'   ✅ {out_thumb}')

shutil.rmtree(tmp, ignore_errors=True)
print('\n✅ All done! Run Cell 4 to download your files.')

In [ ]:
# ── CELL 4: Download to your phone ────────────────────────────────────────────
from google.colab import files
from pathlib import Path

safe = ''.join(c if c.isalnum() else '_' for c in TOPIC)[:35].lower()
mp4  = Path(f'/content/{safe}.mp4')
jpg  = Path(f'/content/{safe}_thumbnail.jpg')

if mp4.exists():
    print('Downloading video...')
    files.download(str(mp4))

if jpg.exists():
    print('Downloading thumbnail...')
    files.download(str(jpg))

print('Done! Check your Downloads folder.')